In [ ]:
import ast
import ipynbname
import pandas as pd
import numpy as np
from Functions.AutoCloud import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT import *
from Functions.TedaOptimize_GD import *

FileName = ipynbname.name()
out_path = f'Optimization\\{FileName[:-4]}\\multi\\Optimization.csv'
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')
sig = HI['PC1'].values

In [ ]:
df = Optimize(FileName=FileName[:-4],OptDim=2,OptSampler='none',OptPrune=False,
                         n_study=1,timeout=60,n_trials=5e1,patience=None,
                         mS=[2.0,4.5],nRS=[1,70],mdS=[1,1],actS=[0,1])

In [ ]:
for i in range(1,6):
    df = Optimize(FileName=FileName[:-4],OptDim=1,OptSampler='tpe',OptPrune=True,
                         n_study=6,timeout=1680,n_trials=1e4,patience=1e3,
                         mS=[2.0,4.5],nLS=[i,i],nRS=[1,70],mdS=[1,1],actS=[0,1])

In [ ]:
#df = df[(df.iloc[:,0] <= 0.07)]
params_list = df.values[:,-11:]
df

In [ ]:
tedas = []
for i,params in enumerate(params_list):
    m,nI,nR,nO,mO,N1,N2,N3,τ,mode,act = params
    X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
    if isinstance(nR, str): nR = ast.literal_eval(nR)

    teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1,N2,N3],mode=mode,act=act,
                tau=τ,rho=0.001,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
    for j,_ in enumerate(X[:]):
        teda.run(X[j])
        teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
        teda.Adapt(Y[j],Z[j])

    teda.c = np.append(teda.c,teda.gm)  
    tedas.append(teda)
    #PlotSeriesPLY(ySeries=[teda.wape_HI_hist,teda.wape_RUL_hist])

In [ ]:
PlotDSI_3D_PLT(tedas[-1])